# ListOps Dataset Exploration

## Dependencies

In [72]:
import numpy as np
import pandas as pd
import re

## Overview

In [73]:
train = pd.read_parquet('../data/raw/train-00000-of-00001.parquet')
train.head()

,Source,Target
0,[MIN 1 [SM 3 6 4 0 5 ] 5 [MED 6 3 8 ] 0 ],0
1,[MIN 1 [MIN 1 7 6 3 ] 1 2 4 4 1 8 3 ],1
2,[MED [MAX 9 7 9 4 6 5 1 4 0 5 ] 3 4 9 [MAX 2 9...,8
3,[MAX [MED 7 [MAX [MAX 6 4 3 1 6 ] [MAX 9 8 1 1...,9
4,[MAX 7 [MIN 6 6 9 8 2 8 ] 9 3 [MIN 4 7 1 5 0 4...,9


In [74]:
regex_pat = r'[A-Z]+|\d+|[\[\](),]'

# Max length of the string sequence
string_seq_len = train['Source'].str.len()
max_string_len, max_string_idx = string_seq_len.max(), string_seq_len.argmax()
print('max_string_len:', max_string_len)
print('max_string_idx:', max_string_idx)
print('max_string:', train['Source'][max_string_idx])
# Note there are blank space characters that can be ommited
print('max_string_len, no spaces:', len(re.findall(regex_pat, train['Source'][max_string_idx])))

max_string_len: 85
max_string_idx: 57509
max_string: [MIN [MED 0 8 ] 8 [MAX 3 [MAX 7 3 ] [MED [MAX 9 8 ] [MAX 9 [MAX 2 5 2 5 ] ] ] 5 4 ] ]
max_string_len, no spaces: 39


## Tokenization strategy

We want to segregate the strings into semantic units - operators, numbers and brackets. 

In [75]:
# Separating a string into semantic tokens
sample_string = train['Source'][420]
print(f'Sample string: {sample_string}')

tokens = re.findall(regex_pat, sample_string)
print(f'Tokens: {tokens}')

Sample string: [MAX 7 [MED 5 [MED 4 5 1 1 9 8 2 ] 9 2 [MED 8 5 8 0 ] [SM 3 4 ] ] 4 6 2 ]
Tokens: ['[', 'MAX', '7', '[', 'MED', '5', '[', 'MED', '4', '5', '1', '1', '9', '8', '2', ']', '9', '2', '[', 'MED', '8', '5', '8', '0', ']', '[', 'SM', '3', '4', ']', ']', '4', '6', '2', ']']


In [85]:
# Applying segregation to the whole dataframe
X = pd.DataFrame(
    train['Source'].str.findall(regex_pat).to_list()
)
X.fillna('nan', inplace=True)
X.head()

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,[,MIN,1,[,SM,3,6,4,0,5,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,[,MIN,1,[,MIN,1,7,6,3,],...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,[,MED,[,MAX,9,7,9,4,6,5,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
3,[,MAX,[,MED,7,[,MAX,[,MAX,6,...,[,MAX,4,1,9,],4,4,],nan
4,[,MAX,7,[,MIN,6,6,9,8,2,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [86]:
# Evaluating vocabulary size
vocab = sorted(set(X.values.ravel()))
print('Vocabulary size:', len(vocab))
print('Vocabulary:', vocab)

Vocabulary size: 17
Vocabulary: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'MAX', 'MED', 'MIN', 'SM', '[', ']', 'nan']


In [87]:
# Tokenizing the whole vector - inputs
# Because of the EOF token, we need at least 17 bits to encode the whole vocabulary
dictionary = {token: idx for idx, token in enumerate(vocab)}
tkX = X.replace(dictionary)
tkX.head()

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,14,12,1,14,13,3,6,4,0,5,...,16,16,16,16,16,16,16,16,16,16
1,14,12,1,14,12,1,7,6,3,15,...,16,16,16,16,16,16,16,16,16,16
2,14,11,14,10,9,7,9,4,6,5,...,16,16,16,16,16,16,16,16,16,16
3,14,10,14,11,7,14,10,14,10,6,...,14,10,4,1,9,15,4,4,15,16
4,14,10,7,14,12,6,6,9,8,2,...,16,16,16,16,16,16,16,16,16,16
